# 09_compare_export.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports & load all model metrics

In [1]:
import pandas as pd
import numpy as np
import joblib, json, os, shutil
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    f1_score, precision_recall_curve, auc,
    roc_auc_score, classification_report
)
import warnings
warnings.filterwarnings("ignore")

COINS = ["Bitcoin","Ethereum","Dogecoin"]
MODEL_TYPES = ["lr","rf","xgb","lgb"]
MODEL_NAMES = {
    "lr":"Logistic Regression",
    "rf":"Random Forest",
    "xgb":"XGBoost",
    "lgb":"LightGBM"
}

def load_feature_cols(coin: str):
    path = Path(f"feature_cols_{coin.lower()}.csv")
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run Notebook 3 first.")
    return pd.read_csv(path)["feature"].tolist()

def time_split(df, val_frac=0.15, test_frac=0.15):
    n = len(df)
    return (
        df.iloc[:int(n*(1-val_frac-test_frac))].copy(),
        df.iloc[int(n*(1-val_frac-test_frac)):int(n*(1-test_frac))].copy(),
        df.iloc[int(n*(1-test_frac)):].copy()
    )


## Cell 2 — Re-evaluate ALL models on test sets

In [2]:
all_results = []

for coin in COINS:
    feat_cols = load_feature_cols(coin)
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    _, val, test = time_split(df)

    X_val = val[feat_cols].values
    y_val = val["target"].values.astype(int)
    X_test = test[feat_cols].values
    y_test = test["target"].values.astype(int)

    for mt in MODEL_TYPES:
        fpath = f"{mt}_model_{coin.lower()}.joblib"
        try:
            model = joblib.load(fpath)
        except FileNotFoundError:
            print(f"⚠️  {fpath} not found — skip")
            continue

        val_prob = model.predict_proba(X_val)[:, 1]
        best_thr = max(
            np.linspace(0.3, 0.7, 41),
            key=lambda t: f1_score(y_val, (val_prob >= t).astype(int))
        )

        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_thr).astype(int)

        prec, rec, _ = precision_recall_curve(y_test, test_prob)
        pr_auc = auc(rec, prec)
        roc = roc_auc_score(y_test, test_prob)
        f1_up = f1_score(y_test, y_pred)

        all_results.append({
            "coin": coin,
            "model_type": mt,
            "model_name": MODEL_NAMES[mt],
            "n_features": len(feat_cols),
            "f1_up": f1_up,
            "roc_auc": roc,
            "pr_auc": pr_auc,
            "threshold": best_thr,
        })
        print(
            f"{coin:12s} {MODEL_NAMES[mt]:22s} "
            f"F1={f1_up:.4f}  ROC={roc:.4f}  PR-AUC={pr_auc:.4f}  thr={best_thr:.2f}"
        )

results_df = pd.DataFrame(all_results)
print("\n=== Full Results Table ===")
print(results_df.to_string())
results_df.to_csv("all_model_results.csv", index=False)


Bitcoin      Logistic Regression    F1=0.7024  ROC=0.5288  PR-AUC=0.5765  thr=0.30
Bitcoin      Random Forest          F1=0.7037  ROC=0.5095  PR-AUC=0.5557  thr=0.30
Bitcoin      XGBoost                F1=0.6154  ROC=0.4838  PR-AUC=0.5516  thr=0.30
Bitcoin      LightGBM               F1=0.7037  ROC=0.5089  PR-AUC=0.5404  thr=0.30
Ethereum     Logistic Regression    F1=0.6954  ROC=0.5045  PR-AUC=0.5764  thr=0.39
Ethereum     Random Forest          F1=0.7249  ROC=0.4973  PR-AUC=0.5891  thr=0.34
Ethereum     XGBoost                F1=0.6512  ROC=0.4980  PR-AUC=0.5842  thr=0.30
Ethereum     LightGBM               F1=0.6375  ROC=0.4741  PR-AUC=0.5857  thr=0.30
Dogecoin     Logistic Regression    F1=0.2239  ROC=0.5525  PR-AUC=0.4836  thr=0.30
Dogecoin     Random Forest          F1=0.5981  ROC=0.5171  PR-AUC=0.4587  thr=0.31
Dogecoin     XGBoost                F1=0.5517  ROC=0.5762  PR-AUC=0.5359  thr=0.30
Dogecoin     LightGBM               F1=0.5244  ROC=0.5838  PR-AUC=0.5275  thr=0.30

===

## Cell 3 — Model comparison bar chart

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ["f1_up","roc_auc","pr_auc"]
metric_labels = ["F1 (Up)", "ROC-AUC", "PR-AUC"]

for ax, coin in zip(axes, COINS):
    g = results_df[results_df["coin"]==coin].sort_values("f1_up", ascending=True)
    colors_bar = ["#3b82f6","#22c55e","#f59e0b","#a855f7"]
    bars = ax.barh(g["model_name"], g["f1_up"], color=colors_bar[:len(g)], edgecolor="white")
    for bar, val in zip(bars, g["f1_up"]):
        ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)
    ax.set_title(f"{coin}\nF1 Score Comparison")
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1 Score (Up class)")
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)

plt.suptitle("Model Comparison — F1 Score per Coin", fontsize=14)
plt.tight_layout()
plt.savefig("model_comparison_f1.png", dpi=150)
plt.show()

## Cell 4 — Heatmap: all metrics × all models × all coins

In [4]:
pivot = results_df.pivot_table(index=["coin","model_name"], values=["f1_up","roc_auc","pr_auc"])
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", ax=ax,
            linewidths=0.5, cbar_kws={"shrink":0.8})
ax.set_title("Model Performance Heatmap (all coins × all models)", fontsize=12)
plt.tight_layout()
plt.savefig("model_heatmap.png", dpi=150)
plt.show()

## Cell 5 — Select BEST model per coin & export deployment package

In [5]:
best_per_coin = {}
for coin in COINS:
    g = results_df[results_df["coin"] == coin].copy()
    if g.empty:
        continue

    # Primary selection metric = class-1 F1, secondary = PR-AUC
    g = g.sort_values(["f1_up", "pr_auc", "roc_auc"], ascending=[False, False, False])
    best_row = g.iloc[0]
    best_per_coin[coin] = best_row
    print(
        f"{coin:12s} → BEST: {best_row['model_name']:22s}  "
        f"F1={best_row['f1_up']:.4f}  PR-AUC={best_row['pr_auc']:.4f}  ROC-AUC={best_row['roc_auc']:.4f}"
    )

# Save 3 final models
for coin in COINS:
    if coin not in best_per_coin:
        continue
    row = best_per_coin[coin]
    mt = row["model_type"]
    src = f"{mt}_model_{coin.lower()}.joblib"
    dst = f"best_model_{coin.lower()}.joblib"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {src} → {dst}")

# Save per-coin feature columns for deployment
feature_map = {}
for coin in COINS:
    feat_cols = load_feature_cols(coin)
    feature_map[coin] = feat_cols
    pd.DataFrame({"feature": feat_cols}).to_csv(f"best_feature_cols_{coin.lower()}.csv", index=False)

with open("best_feature_cols_by_coin.json", "w", encoding="utf-8") as f:
    json.dump(feature_map, f, indent=2)

# Save per-coin metadata JSON
metadata = {}
for coin in COINS:
    if coin not in best_per_coin:
        continue
    row = best_per_coin[coin]
    mt = row["model_type"]
    metadata[coin] = {
        "pipeline": MODEL_NAMES[mt],
        "model_type": mt,
        "coin": coin,
        "threshold": float(row["threshold"]),
        "f1_class1": float(row["f1_up"]),
        "roc_auc": float(row["roc_auc"]),
        "pr_auc": float(row["pr_auc"]),
        "n_features": int(row["n_features"]),
        "feature_file": f"best_feature_cols_{coin.lower()}.csv",
    }

with open("best_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\n✅ Saved best_model_metadata.json")
print("✅ Saved best_feature_cols_by_coin.json")
print(json.dumps(metadata, indent=2))


Bitcoin      → BEST: Random Forest           F1=0.7037  PR-AUC=0.5557  ROC-AUC=0.5095
Ethereum     → BEST: Random Forest           F1=0.7249  PR-AUC=0.5891  ROC-AUC=0.4973
Dogecoin     → BEST: Random Forest           F1=0.5981  PR-AUC=0.4587  ROC-AUC=0.5171
Copied rf_model_bitcoin.joblib → best_model_bitcoin.joblib
Copied rf_model_ethereum.joblib → best_model_ethereum.joblib
Copied rf_model_dogecoin.joblib → best_model_dogecoin.joblib

✅ Saved best_model_metadata.json
✅ Saved best_feature_cols_by_coin.json
{
  "Bitcoin": {
    "pipeline": "Random Forest",
    "model_type": "rf",
    "coin": "Bitcoin",
    "threshold": 0.3,
    "f1_class1": 0.7037433155080214,
    "roc_auc": 0.5094971086214654,
    "pr_auc": 0.5556793295755862,
    "n_features": 60,
    "feature_file": "best_feature_cols_bitcoin.csv"
  },
  "Ethereum": {
    "pipeline": "Random Forest",
    "model_type": "rf",
    "coin": "Ethereum",
    "threshold": 0.33999999999999997,
    "f1_class1": 0.7248908296943232,
    "roc_auc

## Cell 6 — Build deploy_demo_rows.csv (one per coin per available test date)

In [7]:
demo_frames = []

for coin in COINS:
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"]).sort_values("date")
    _, _, test = time_split(df)

    # copy so we do not modify the sliced dataframe in-place
    test = test.copy()

    # app expects these two columns
    test["coin"] = coin
    test["date"] = pd.to_datetime(test["date"]).dt.strftime("%Y-%m-%d")

    # app expects actual_target, not target
    if "target" in test.columns:
        test["actual_target"] = test["target"].astype(int)
    elif "actual_target" in test.columns:
        test["actual_target"] = test["actual_target"].astype(int)
    else:
        raise ValueError(f"No target/actual_target column found for {coin}")

    # put key deployment columns first
    front_cols = ["coin", "date", "actual_target"]
    remaining_cols = [c for c in test.columns if c not in front_cols + ["target"]]
    test = test[front_cols + remaining_cols]

    demo_frames.append(test)

deploy_demo_rows = pd.concat(demo_frames, ignore_index=True)

# final safety checks
required_cols = ["coin", "date", "actual_target"]
missing = [c for c in required_cols if c not in deploy_demo_rows.columns]
if missing:
    raise ValueError(f"deploy_demo_rows is missing required columns: {missing}")

# save WITH headers
deploy_demo_rows.to_csv("deploy_demo_rows.csv", index=False)

print(f"Saved deploy_demo_rows.csv  shape={deploy_demo_rows.shape}")
print("\nColumns:")
print(deploy_demo_rows.columns.tolist())

print("\nCoin counts:")
print(deploy_demo_rows["coin"].value_counts())

print("\nPreview:")
print(deploy_demo_rows[["coin", "date", "actual_target"]].head())

Saved deploy_demo_rows.csv  shape=(1132, 63)

Columns:
['coin', 'date', 'actual_target', 'return_1', 'return_3', 'return_7', 'return_14', 'return_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_min_7', 'rolling_max_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_min_14', 'rolling_max_14', 'rolling_mean_30', 'rolling_std_30', 'rolling_min_30', 'rolling_max_30', 'price_vs_ma7', 'price_vs_ma14', 'price_vs_ma30', 'range', 'upper_shadow', 'lower_shadow', 'body_size', 'atr_14', 'rsi_14', 'rsi_7', 'macd', 'macd_signal', 'macd_diff', 'adx_14', 'cci_20', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct', 'volume_change', 'volume_ma7', 'volume_vs_ma7', 'obv', 'obv_change', 'quarter', 'is_month_end', 'is_month_start', 'lag1_return', 'lag2_return', 'lag3_return', 'btc_twitter_sentiment_mean', 'btc_twitter_sentiment_std', 'btc_twitter_tweet_count', 'btc_twitter_sentiment_ma3', 'btc_twitter_sentiment_chg3', 'btc_twitter_tweet_count_ma3', 'btc_twitter_sentiment_ma7', 'btc_twitter_sentiment_chg7', 'btc_